# Scheme B vs C: Performance Comparison

Runs both approaches on the **same** config and data:
- **Scheme B**: Regression (SmoothL1) → scalar K estimate
- **Scheme C**: Classification (CrossEntropy) → K classes 0..Kmax

Compares: val MAE, accuracy (C only), and confusion-style diagnostics.


In [ ]:
# Assumes: pip install -e . from project root

In [ ]:
### Imports
import numpy as np
import torch
import matplotlib.pyplot as plt
from spectackle.config import deep_update, set_cpu_safety
from spectackle.data import BASE_CFG, make_loaders
from spectackle.models import CountNet1D, CountNet1D_Classify
from spectackle.training import train_scheme_b, train_scheme_c
from spectackle.plotting import collect_predictions_bc
set_cpu_safety(1)  ### Avoid CPU oversubscription on macOS

In [ ]:
### Config + data (default poisson K distribution)
cfg = deep_update(BASE_CFG, {})
Kmax = int(cfg["max_components"])
train_loader, val_loader = make_loaders(
    cfg, n_train=50_000, n_val=5_000, bs_train=128, bs_val=256
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kmax={Kmax}, device={device}")

In [ ]:
### Run both schemes on the same data

print("=== Scheme B (regression) ===")
model_b = CountNet1D(width=64)
model_b = train_scheme_b(
    model_b, train_loader, val_loader,
    device=device, lr=1e-3, epochs=3, log_every=200, Kmax=Kmax,
)

print("\n=== Scheme C (classification) ===")
model_c = CountNet1D_Classify(Kmax=Kmax, width=64)
model_c = train_scheme_c(
    model_c, train_loader, val_loader,
    device=device, lr=1e-3, epochs=3, log_every=200,
)

In [ ]:
### Collect predictions
y_true, y_b, y_c_argmax, y_c_exp = collect_predictions_bc(
    model_b, model_c, val_loader, device, Kmax
)

In [ ]:
### Summary metrics

mae_b        = np.abs(y_b - y_true).mean()
mae_c_argmax = np.abs(y_c_argmax - y_true).mean()
mae_c_exp    = np.abs(y_c_exp - y_true).mean()
acc_c        = (y_c_argmax == y_true).mean()
bias_b       = (y_b - y_true).mean()
bias_c       = (y_c_argmax - y_true).mean()

print("=== Performance Summary (same val set) ===")
print(f"Scheme B (regression, rounded):  MAE = {mae_b:.3f}  bias = {bias_b:+.3f}")
print(f"Scheme C (argmax):               MAE = {mae_c_argmax:.3f}  bias = {bias_c:+.3f}  Accuracy = {acc_c:.3f}")
print(f"Scheme C (E[K] rounded):          MAE = {mae_c_exp:.3f}")

In [ ]:
### Comparison plots

edges = np.arange(-0.5, Kmax + 1.5, 1.0)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

H_b, _, _ = np.histogram2d(y_true, y_b, bins=[edges, edges])
axes[0].imshow(H_b.T, origin="lower", aspect="equal",
               extent=[edges[0], edges[-1], edges[0], edges[-1]])
axes[0].plot([0, Kmax], [0, Kmax], "k-", lw=1.5)
axes[0].set_xlabel("K true"); axes[0].set_ylabel("K pred")
axes[0].set_title(f"Scheme B (MAE={mae_b:.3f})")
axes[0].set_xticks(np.arange(0, Kmax + 1)); axes[0].set_yticks(np.arange(0, Kmax + 1))

H_c, _, _ = np.histogram2d(y_true, y_c_argmax, bins=[edges, edges])
axes[1].imshow(H_c.T, origin="lower", aspect="equal",
               extent=[edges[0], edges[-1], edges[0], edges[-1]])
axes[1].plot([0, Kmax], [0, Kmax], "k-", lw=1.5)
axes[1].set_xlabel("K true"); axes[1].set_ylabel("K pred")
axes[1].set_title(f"Scheme C argmax (MAE={mae_c_argmax:.3f}, acc={acc_c:.3f})")
axes[1].set_xticks(np.arange(0, Kmax + 1)); axes[1].set_yticks(np.arange(0, Kmax + 1))

err_b = y_b - y_true
err_c = y_c_argmax - y_true
bins = np.arange(err_b.min() - 0.5, err_b.max() + 1.5, 1)
axes[2].hist(err_b, bins=bins, alpha=0.5,
             label=f"B (MAE={mae_b:.3f})", color="C0", density=True)
axes[2].hist(err_c, bins=bins, alpha=0.5,
             label=f"C (MAE={mae_c_argmax:.3f})", color="C1", density=True)
axes[2].axvline(0, color="k", ls="--", lw=1)
axes[2].set_xlabel("K_pred - K_true"); axes[2].set_ylabel("density")
axes[2].set_title("Error distribution"); axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
### MAE vs K_true

ks = np.arange(0, Kmax + 1)
mae_b_by_k = [np.abs(y_b[y_true == k] - k).mean() if np.any(y_true == k) else np.nan for k in ks]
mae_c_by_k = [np.abs(y_c_argmax[y_true == k] - k).mean() if np.any(y_true == k) else np.nan for k in ks]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ks, mae_b_by_k, "o-", label="Scheme B", color="C0")
ax.plot(ks, mae_c_by_k, "s-", label="Scheme C (argmax)", color="C1")
ax.set_xlabel("K true")
ax.set_ylabel("MAE")
ax.set_title("MAE vs K_true")
ax.legend()
ax.set_xticks(ks)
plt.tight_layout()
plt.show()

In [ ]:
### K count accuracy

rows = []
for k in ks:
    m = y_true == k
    n = m.sum()
    if n == 0:
        rows.append((k, 0, np.nan, np.nan, np.nan))
    else:
        mae_bk = np.abs(y_b[m] - k).mean()
        mae_ck = np.abs(y_c_argmax[m] - k).mean()
        acc_ck = (y_c_argmax[m] == k).mean()
        rows.append((k, int(n), mae_bk, mae_ck, acc_ck))

print(f"{'K':>2}  {'n':>5}  {'MAE_B':>7}  {'MAE_C':>7}  {'acc_C':>7}")
print("-" * 35)
for k, n, mb, mc, ac in rows:
    mb_s = f"{mb:.3f}" if not np.isnan(mb) else "   -"
    mc_s = f"{mc:.3f}" if not np.isnan(mc) else "   -"
    ac_s = f"{ac:.3f}" if not np.isnan(ac) else "   -"
    print(f"{k:>2}  {n:>5}  {mb_s:>7}  {mc_s:>7}  {ac_s:>7}")

In [ ]:
### Agreement / disagreement: when do B and C agree, and which does better?

agree = y_b == y_c_argmax
n_agree = agree.sum()
n_disagree = (~agree).sum()

print("=== Agreement ===")
print(f"B and C same prediction: {n_agree} / {len(y_true)} ({n_agree/len(y_true):.2%})")

if n_agree > 0:
    acc_when_agree = (y_b[agree] == y_true[agree]).mean()
    print(f"  When they agree: accuracy = {acc_when_agree:.3f}")

if n_disagree > 0:
    b_right = (y_b[~agree] == y_true[~agree]).sum()
    c_right = (y_c_argmax[~agree] == y_true[~agree]).sum()
    both_wrong = ((y_b[~agree] != y_true[~agree]) & (y_c_argmax[~agree] != y_true[~agree])).sum()
    print(f"\nWhen they disagree ({n_disagree} cases):")
    print(f"  B right, C wrong: {b_right};  C right, B wrong: {c_right};  both wrong: {both_wrong}")